In [1]:
import pytest
import ipytest
from room import *

In [2]:
ipytest.autoconfig()

In [3]:
%%ipytest

%%ipytest

import numpy as np
import pytest

# ==========================================
# TEST SUITE FOR ROOM CLASS
# ==========================================

@pytest.fixture
def complex_design():
    """Fixture providing a standard room with special surfaces."""
    return {
        'environment': {
            'dimensions': [5.0, 4.0, 3.0], # L, W, H
            'wall_resolution': (10, 10),   # 100 elements per wall
            'reflectivity': {
                'floor': 0.2,
                'ceiling': 0.8,
                'walls': 0.5
            },
            'special_surfaces': [
                {
                    'type': 'window', 
                    'dims': (1, 1), 
                    'center': [2.5, 4.0, 1.5], # Middle of North Wall
                    'const_axis': 1, 
                    'resolution': (5, 5), 
                    'normal': [0, -1, 0], 
                    'reflectivity': 0.1, 
                    'name': 'Main_Window'
                },
                {
                    'type': 'RIS', 
                    'dims': (0.5, 0.5), 
                    'center': [0.0, 2.0, 1.5], # Middle of West Wall
                    'const_axis': 0, 
                    'resolution': (10, 10), 
                    'normal': [1, 0, 0], 
                    'reflectivity': 0.95, 
                    'name': 'RIS_Unit'
                }
            ]
        }
    }

# --- 1. Assembly Tests ---

def test_room_shell_integrity(complex_design):
    """Verifies that all 6 walls are correctly instantiated and merged."""
    rb = RoomBuilder(complex_design)
    room = Room(rb, ignore_RIS=True, ignore_windows=True)
    
    # Check wall count
    assert len(room.walls) == 6
    
    # Verify master batch size (6 walls * 10x10 resolution = 600)
    assert room.Tx_wall_elements.N == 600
    assert room.Rx_wall_elements.N == 600
    
    # Check coordinate bounds (Ceiling should be at H=3.0, Floor at Z=0.0)
    assert np.all(room.ceiling.r_surface[:, 2] == 3.0)
    assert np.all(room.floor.r_surface[:, 2] == 0.0)

# --- 2. Special Surface Logic Tests ---

def test_surface_type_segregation(complex_design):
    """Ensures windows and RIS elements are stored in separate master batches."""
    rb = RoomBuilder(complex_design)
    room = Room(rb)
    
    # Windows
    assert len(room.windows) == 1
    assert room.Tx_windows_elements is not None
    assert room.Tx_windows_elements.N == 25 # 5x5 resolution
    
    # RIS
    assert len(room.RIS) == 1
    assert room.Tx_RIS_elements is not None
    assert room.Tx_RIS_elements.N == 100 # 10x10 resolution

def test_reflectivity_masking_accuracy(complex_design):
    """Verifies that wall tiles behind special surfaces are zeroed out."""
    rb = RoomBuilder(complex_design)
    room = Room(rb)
    
    # Check total zeroed-out reflectivity patches
    # The Window is 1x1m on a 5x3m wall (10x10 res). Wall tiles are 0.5x0.3m.
    # The RIS is 0.5x0.5m on a 4x3m wall (10x10 res). Wall tiles are 0.4x0.3m.
    zeros = np.sum(room.Rx_wall_elements.refl == 0)
    
    assert zeros > 0
    # Expected: Window (~4-6 tiles) + RIS (~4-6 tiles)
    assert 8 <= zeros <= 20 

# --- 3. Incremental Addition Tests ---

def test_cumulative_add_surface(complex_design):
    """Tests that calling add_surface repeatedly accumulates transmitters correctly."""
    rb = RoomBuilder(complex_design)
    room = Room(rb, ignore_windows=True, ignore_RIS=True) # Start empty
    
    # Manually add two windows
    win_spec = {
        'center': [2.5, 4.0, 1.5], 'dims': (1,1), 'const_axis': 1, 
        'resolution': (2,2), 'type': 'window'
    }
    
    room.add_surface(Surface(**win_spec, name="Win_A"))
    assert room.Tx_windows_elements.N == 4
    
    room.add_surface(Surface(**win_spec, name="Win_B"))
    # Total should now be 8 (using the __add__ logic of Elements)
    assert room.Tx_windows_elements.N == 8

def test_ignore_flags_functionality(complex_design):
    """Verifies that the ignore_RIS and ignore_windows flags work as intended."""
    rb = RoomBuilder(complex_design)
    
    room = Room(rb, ignore_RIS=True, ignore_windows=True)
    assert len(room.windows) == 0
    assert len(room.RIS) == 0
    assert room.Tx_windows_elements is None
    assert room.Tx_RIS_elements is None

.....                                                                                        [100%]
5 passed in 0.04s
.....                                                                                        [100%]
5 passed in 0.03s
